In [29]:
import os,csv
import sys
from scipy import stats
sys.path.append('../include')
from load_dictionary import load_dictionary
import db_init as db
import pandas as pd
from collections import defaultdict

In [28]:

removed_id = set([8433,202468,211750,201798,201791,35150,174899,240710,8619,199415,170326,190790])

dicttimestamp = '20180706'
engine = db.connectToPostgreSQL(server='cns-postgres-myaura')
tablename = 'dictionaries.dict_%s' % (dicttimestamp)
sql = """
    SELECT
        d.id,
        COALESCE(d.id_parent,d.id) AS id_parent,
        d.dictionary,
        d.token,
        COALESCE(p.token, d.token) as parent,
        d.type,
        d.source,
        d.id_original,
        COALESCE(p.id_original, d.id_original) as id_original_parent
        FROM %s d
        LEFT JOIN %s p ON d.id_parent = p.id
        """ % (tablename, tablename)
dfD = pd.read_sql(sql, engine, index_col='id')

In [20]:
before_file = '../instagram/tmp-data/04-instagram-epilepsy-network-20180706-samepost-edges.csv'
after_file = '../instagram/tmp-data/04-instagram-epilepsy-network-20210321-samepost-edges.csv'

In [24]:
def read_csv(filepath):
    true_set = set()
    false_set = set()
    with open(filepath) as fp:
        for line in csv.DictReader(fp):
            source = int(line['source'])
            target = int(line['target'])
            if source in removed_id or target in removed_id:
                continue
            if line['is_metric'] == 'True':
                true_set.add(frozenset([source, target]))
            else:
                false_set.add(frozenset([source, target]))
    return true_set,false_set

In [25]:
old_true, old_false = read_csv(before_file)
new_true, new_false = read_csv(after_file)

In [26]:
print(len(old_true), len(old_false))
print(len(new_true), len(new_false))
print(len(new_true - old_true))
print(len(old_true - new_true))

4719 22702
4789 21165
176
106


In [37]:
def get_token(idx):
    return dfD.loc[int(idx)]['token']
def show_aggregate(pair_set):
    count = defaultdict(int)
    for a,b in pair_set:
        count[a] += 1
        count[b] += 1
    sorted_list = sorted(count.items(), key=lambda x:x[1], reverse=True)
    for i in range(10):
        tid = sorted_list[i][0]
        token = get_token(tid)
        print(tid, token, sorted_list[i][1])

In [36]:
show_aggregate(new_true-old_true)
print('-----------')
show_aggregate(old_true-new_true)

815 Diazepam 14
187288 Trance 10
176626 Feeling hot 9
169842 Adoption 6
8604 Grapefruit 5
178293 Homicide 5
173556 Chills 5
179006 Influenza 4
178464 Hyperhidrosis 4
182630 Pain 4
-----------
815 Diazepam 15
181534 Nasopharyngitis 14
176626 Feeling hot 11
175013 Dependence 9
176223 Euphoric mood 6
174071 Confusional state 3
8595 Coffee bean 3
8599 Egg 3
461 Fluoxetine 2
414 Zolpidem 2


In [38]:
def find_token_pair(pair_set, tid):
    for a,b in pair_set:
        if a == tid or b == tid:
            print(a, get_token(a), b, get_token(b))

In [39]:
find_token_pair(new_true - old_true, 815)

1035 Ibuprofen 815 Diazepam
179162 Insomnia 815 Diazepam
179084 Injection 815 Diazepam
774 Naproxen 815 Diazepam
1492 Bromazepam 815 Diazepam
669 Midazolam 815 Diazepam
222 Temazepam 815 Diazepam
414 Zolpidem 815 Diazepam
559 Atropine 815 Diazepam
1052 Clonazepam 815 Diazepam
173774 Claustrophobia 815 Diazepam
10185 Benzodiazepine 815 Diazepam
1208 Quetiapine 815 Diazepam
185593 Sciatica 815 Diazepam


In [40]:
find_token_pair(old_true - new_true, 815)

942 Hydrocodone 815 Diazepam
5778 Stanozolol 815 Diazepam
9951 Nefopam 815 Diazepam
8595 Coffee bean 815 Diazepam
187194 Torticollis 815 Diazepam
8154 Dexketoprofen 815 Diazepam
239925 Kratom 815 Diazepam
240914 Cannabis 815 Diazepam
646 Acamprosate 815 Diazepam
884 Ethanol 815 Diazepam
170661 Anxiety 815 Diazepam
5750 Nitrous oxide 815 Diazepam
307 Acetaminophen 815 Diazepam
1096 Cefuroxime 815 Diazepam
182586 Overdose 815 Diazepam
